# Machine Learning in Python - Project 

Due Friday, Apr 10th by 4 pm.

*Include contributors names here*

## Setup

*Install any packages here, define any functions if neeed, and load data*

In [1]:
# Add any additional libraries or submodules below

# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn

In [2]:
# Load data  
df = pd.read_csv("unicef_malawi.csv")
df.head()

,HH1,HH2,LN,FS4,CB3,CB4,CB5A,CB5B,CB7,CB11,...,HC19,TN1,WS1,WS3,WS4,WS7,WS11,WS14,WS15,HW5
0,1.0,2.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,NO,NO,PIPED WATER: PUBLIC TAP / STANDPIPE,ELSEWHERE,5.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,ELSEWHERE,YES,NO
1,1.0,3.0,1.0,1.0,5.0,YES,ECE,NaN,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITHOUT SLAB / OPEN PIT,IN OWN YARD / PLOT,NO,YES
2,1.0,4.0,2.0,2.0,16.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,6.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,YES
3,1.0,8.0,2.0,2.0,13.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO
4,1.0,10.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,8.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO


# Introduction

*This section should include a brief introduction to the task and the data (assume this is a report you are delivering to a professional body (e.g. UNICEF)).*

*Briefly outline the approaches being used and the conclusions that you are able to draw.*

# Exploratory Data Analysis and Feature Engineering

*Include a detailed discussion of the data with a particular emphasis on the features of the data that are relevant for the subsequent modeling. Including visualizations of the data is strongly encouraged - all code and plots must also be described in the write up. Think carefully about whether each plot needs to be included in your final draft and the appropriate type of plot and summary for each variable type - your report should include figures but they should be as focused and impactful as possible.*

*You should also split your data into training and testing sets, ideally before you look to much into the features and relationships with the target.*

*Additionally, this section should also motivate and describe any preprocessing / feature engineering of the data. Pipelines should be used and feature engineering steps that are be performed as part of an sklearn pipeline can be mentioned here but should be implemented in the following section.*

*All code and figures should be accompanied by text that provides an overview / context to what is being done or presented.*

In [2]:
import pandas as pd
import numpy as np

# read data
df = pd.read_csv("unicef_malawi_cleaned_full.csv")

# Transform into binary variable
df["child_depression"] = df["FCF26"].map({
    "No": 0,
    "YES": 1
})

# check results
print(df[["FCF26", "child_depression"]].head())
print(df["child_depression"].value_counts(dropna=False))
df = df.dropna(subset=["child_depression"]).copy()
print(df["child_depression"].isna().sum())
print(df.shape)


  FCF26  child_depression
0    No               0.0
1    No               0.0
2    No               0.0
3    No               0.0
4    No               0.0
child_depression
1.0    7131
0.0    5905
NaN     126
Name: count, dtype: int64
0
(13036, 91)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# =========================================================
# 0. 基础设置
# =========================================================

target_col = "child_depression"

missing_thresh = 0.30   # 自变量缺失率阈值
mi_thresh = 0.001       # 与 y 的最小信息量阈值，可调
corr_thresh = 0.70      # 区块内去冗余阈值：|Spearman rho| >= 0.70

continuous_score_candidates = [
    "wscore", "WB4", "MA2", "CSURV", "CDEAD"
]

zero_heavy_candidates = ["CL3", "CL13"]

# =========================================================
# 1. 先去掉目标变量缺失
# =========================================================

df = df.copy()


df = df.dropna(subset=[target_col]).copy()

print("=== Shape after dropping missing target ===")
print(df.shape)

# =========================================================
# 2. 定义区块
# =========================================================

blocks = {
    "child_background": [
        "CB3","CB4","CB5A","CB5B","CB7","CB11",
        "HH6","HH7","HL4","ethnicity","wscore"
    ],
    
    "child_labour": [
        "CL2","CL3","CL12","CL13"
    ],
    
    "child_discipline": [
        "FCD2A","FCD2B","FCD2C","FCD2D","FCD2E",
        "FCD2F","FCD2G","FCD2H","FCD2I","FCD2J",
        "FCD2K","FCD5"
    ],
    
    "mother_background": [
        "WB4","WB5","WB6A","WB6B","WB14"
    ],
    
    "domestic_violence": [
        "DV1A","DV1B","DV1C","DV1D","DV1E"
    ],
    
    "victimization": [
        "VT1","VT9","VT20","VT21",
        "VT22A","VT22B","VT22C","VT22D","VT22E","VT22F","VT22X"
    ],
    
    "marriage_union": [
        "MSTATUS","MA2","MA3"
    ],
    
    "adult_functioning": [
        "disability","AF10","AF11","AF12"
    ],
    
    "tobacco_alcohol": [
        "TA1","TA14"
    ],
    
    "life_satisfaction": [
        "LS1","LS2","LS3","LS4"
    ],
    
    "fertility": [
        "CSURV","CDEAD"
    ],
    
    "household_characteristics": [
        "HC4","HC5","HC8","HC11","HC12",
        "HC13","HC14","HC15","HC17","HC19"
    ],
    
    "insecticide_nets": [
        "TN1"
    ],
    
    "water_sanitation": [
        "WS1","WS3","WS4","WS7","WS11","WS14","WS15"
    ],
    
    "handwashing": [
        "HW5"
    ]
}

# =========================================================
# 3. 辅助函数
# =========================================================

def clean_special_missing(series):
    s = series.copy()
    s = s.replace([
        "DK", "NO RESPONSE", "DK / NO OPINION",
        "DON'T KNOW", "MISSING", "NA", "N/A", "", " "
    ], np.nan)
    return s

def to_numeric_safe(series):
    return pd.to_numeric(clean_special_missing(series), errors="coerce")

def infer_feature_type(series, numeric_ratio_thresh=0.8):
    """
    粗略判断变量类型：
    - 大部分非缺失值可转数值 -> numeric
    - 否则 -> categorical
    """
    s = clean_special_missing(series)
    non_missing = s.dropna()

    if len(non_missing) == 0:
        return "categorical"

    converted = pd.to_numeric(non_missing, errors="coerce")
    numeric_ratio = converted.notna().mean()

    if numeric_ratio >= numeric_ratio_thresh:
        return "numeric"
    return "categorical"

def make_ordered_from_zero_heavy(series, new_name):
    """
    适合像 CL3 / CL13 这种 0 值很多、右偏明显的变量。
    分箱规则：
    0 -> 0
    正值部分按三分位分成 1/2/3
    """
    s = to_numeric_safe(series)
    out = pd.Series(np.nan, index=s.index, dtype="float")

    out[s == 0] = 0

    pos = s[s > 0]
    if len(pos) >= 4 and pos.nunique() >= 3:
        try:
            out.loc[pos.index] = pd.qcut(
                pos, q=3, labels=[1, 2, 3], duplicates="drop"
            ).astype(float)
        except Exception:
            q1, q2 = pos.quantile([1/3, 2/3])
            out.loc[(s > 0) & (s <= q1)] = 1
            out.loc[(s > q1) & (s <= q2)] = 2
            out.loc[(s > q2)] = 3
    else:
        out.loc[s > 0] = 1

    return out.rename(new_name)

def make_log_transform(series, new_name):
    s = to_numeric_safe(series)
    s = s.where(s >= 0, np.nan)
    return np.log1p(s).rename(new_name)

def choose_representation(var, df):
    """
    决定变量最终表示形式：
    - raw
    - ordered_from_zero_heavy
    - log1p
    """
    s_num = to_numeric_safe(df[var])
    non_missing = s_num.dropna()

    if len(non_missing) == 0:
        return var, df[var], "raw_non_numeric_or_empty"

    zero_pct = (non_missing == 0).mean()
    skewness = non_missing.skew()

    if var in zero_heavy_candidates and zero_pct >= 0.40:
        new_name = f"{var}_ord"
        return new_name, make_ordered_from_zero_heavy(df[var], new_name), "ordered_from_zero_heavy"

    if var in continuous_score_candidates and pd.notna(skewness) and abs(skewness) >= 1:
        new_name = f"{var}_log1p"
        return new_name, make_log_transform(df[var], new_name), "log1p"

    return var, df[var], "raw"

def encode_single_feature_for_mi(series):
    """
    为 mutual information 计算做编码：
    - numeric: 数值化 + 中位数填补
    - categorical: 众数填补 + ordinal encode
    返回：
        x_array, discrete_flag
    """
    ftype = infer_feature_type(series)

    if ftype == "numeric":
        x = to_numeric_safe(series).to_frame()
        imp = SimpleImputer(strategy="median")
        x_imp = imp.fit_transform(x)
        return x_imp, False

    else:
        x = clean_special_missing(series).astype("object").to_frame()
        imp = SimpleImputer(strategy="most_frequent")
        x_imp = imp.fit_transform(x)
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        x_enc = enc.fit_transform(x_imp)
        return x_enc, True

def compute_mutual_info_scores(data, variables, target_col):
    """
    计算每个变量与目标 y 的 MI 分数
    """
    y = data[target_col].values
    scores = []

    for v in variables:
        try:
            Xv, discrete_flag = encode_single_feature_for_mi(data[v])
            mi = mutual_info_classif(
                Xv, y,
                discrete_features=discrete_flag,
                random_state=42
            )[0]
        except Exception:
            mi = np.nan

        scores.append({
            "variable": v,
            "mi_score": mi,
            "missing_pct": data[v].isna().mean(),
            "feature_type": infer_feature_type(data[v])
        })

    return pd.DataFrame(scores).sort_values(
        by=["mi_score", "missing_pct"],
        ascending=[False, True]
    )

def encode_for_corr(series):
    """
    为区块内 x-x 相关性分析做编码：
    - numeric: 直接数值化
    - YES/NO: 映射为 1/0
    - 其他分类变量: factorize
    """
    s = clean_special_missing(series)

    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")

    s = s.astype(str).str.strip().str.upper()
    s = s.replace({"NAN": np.nan})

    binary_map = {
        "YES": 1, "NO": 0,
        "Y": 1, "N": 0,
        "TRUE": 1, "FALSE": 0
    }

    unique_vals = set(s.dropna().unique())
    if len(unique_vals) > 0 and unique_vals.issubset(set(binary_map.keys())):
        return s.map(binary_map)

    codes, _ = pd.factorize(s, sort=True)
    return pd.Series(codes, index=s.index).replace(-1, np.nan)

def select_block_variables_xy_and_withinblock_corr(
    data,
    vars_list,
    target_col,
    missing_thresh=0.30,
    mi_thresh=0.001,
    corr_thresh=0.70
):
    """
    区块内筛选逻辑：
    1) 先删缺失率 > missing_thresh
    2) 计算每个变量与 y 的 MI
    3) 只保留 MI >= mi_thresh 的变量
    4) 在这些变量中按 MI 从高到低排序
    5) 贪心去冗余：若某变量与区块内已选变量的最大相关性 >= corr_thresh，则删除
    6) 不设每个区块固定上限
    """
    existing_vars = [v for v in vars_list if v in data.columns]

    if len(existing_vars) == 0:
        return {
            "selected": [],
            "dropped_high_missing": [],
            "dropped_low_mi": [],
            "dropped_high_corr": [],
            "mi_table": pd.DataFrame(),
            "corr_matrix": pd.DataFrame()
        }

    # 1. 缺失率筛选
    missing_pct = data[existing_vars].isna().mean()
    candidate_vars = missing_pct[missing_pct <= missing_thresh].index.tolist()
    dropped_high_missing = [v for v in existing_vars if v not in candidate_vars]

    if len(candidate_vars) == 0:
        return {
            "selected": [],
            "dropped_high_missing": dropped_high_missing,
            "dropped_low_mi": [],
            "dropped_high_corr": [],
            "mi_table": pd.DataFrame(),
            "corr_matrix": pd.DataFrame()
        }

    # 2. 计算 x-y 的 MI
    mi_table = compute_mutual_info_scores(data, candidate_vars, target_col)
    mi_table = mi_table.sort_values(by=["mi_score", "missing_pct"], ascending=[False, True])

    # 3. 只保留 MI 达标变量
    mi_candidates = mi_table.loc[mi_table["mi_score"].fillna(0) >= mi_thresh, "variable"].tolist()
    dropped_low_mi = [v for v in candidate_vars if v not in mi_candidates]

    if len(mi_candidates) == 0:
        return {
            "selected": [],
            "dropped_high_missing": dropped_high_missing,
            "dropped_low_mi": dropped_low_mi,
            "dropped_high_corr": [],
            "mi_table": mi_table,
            "corr_matrix": pd.DataFrame()
        }

    # 4. 区块内 x-x 相关性矩阵
    encoded = pd.DataFrame({
        v: encode_for_corr(data[v]) for v in mi_candidates
    })
    corr = encoded.corr(method="spearman").abs()

    ranked_vars = mi_table.loc[
        mi_table["variable"].isin(mi_candidates), "variable"
    ].tolist()

    # 5. 贪心去冗余：按 MI 排序，优先保留更相关 y 的变量
    selected = []
    dropped_high_corr = []

    for v in ranked_vars:
        if len(selected) == 0:
            selected.append(v)
        else:
            max_corr = corr.loc[v, selected].max()
            if pd.isna(max_corr) or max_corr < corr_thresh:
                selected.append(v)
            else:
                dropped_high_corr.append(v)

    return {
        "selected": selected,
        "dropped_high_missing": dropped_high_missing,
        "dropped_low_mi": dropped_low_mi,
        "dropped_high_corr": dropped_high_corr,
        "mi_table": mi_table,
        "corr_matrix": corr
    }

# =========================================================
# 4. 变量表示转换
# =========================================================

all_block_vars = []
for vs in blocks.values():
    all_block_vars.extend(vs)
all_block_vars = list(dict.fromkeys([v for v in all_block_vars if v in df.columns]))

analysis_df = df.copy()
representation_map = {}
transformation_log = []

for v in all_block_vars:
    new_name, new_series, trans_type = choose_representation(v, analysis_df)
    representation_map[v] = new_name

    if new_name != v:
        analysis_df[new_name] = new_series

    transformation_log.append({
        "original_variable": v,
        "used_variable": new_name,
        "transformation": trans_type
    })

transformation_log_df = pd.DataFrame(transformation_log)

print("\n=== Transformation log ===")
print(transformation_log_df)

# =========================================================
# 5. 用转换后的变量重建区块
# =========================================================

transformed_blocks = {}
for block_name, vars_list in blocks.items():
    transformed_blocks[block_name] = [
        representation_map[v] for v in vars_list if v in representation_map
    ]

print("\n=== Transformed blocks ===")
for k, v in transformed_blocks.items():
    print(k, ":", v)

# =========================================================
# 6. 区块内筛选：x-y关系 + 区块内x-x相关性
# =========================================================

selection_results = {}

for block_name, vars_list in transformed_blocks.items():
    result = select_block_variables_xy_and_withinblock_corr(
        data=analysis_df,
        vars_list=vars_list,
        target_col=target_col,
        missing_thresh=missing_thresh,
        mi_thresh=mi_thresh,
        corr_thresh=corr_thresh
    )
    selection_results[block_name] = result

    print("\n" + "="*80)
    print(f"BLOCK: {block_name}")
    print("="*80)

    print("\nSelected variables:")
    print(result["selected"])

    print("\nDropped due to missingness > threshold:")
    print(result["dropped_high_missing"])

    print("\nDropped due to low MI with y:")
    print(result["dropped_low_mi"])

    print("\nDropped due to high within-block correlation:")
    print(result["dropped_high_corr"])

    print("\nMI table:")
    print(result["mi_table"])

    print("\nAbsolute Spearman correlation matrix (within block):")
    print(result["corr_matrix"].round(2))

# =========================================================
# 7. 汇总最终变量
# =========================================================

final_selected_vars = []
for block_name, result in selection_results.items():
    final_selected_vars.extend(result["selected"])

final_selected_vars = list(dict.fromkeys(final_selected_vars))

print("\n" + "="*80)
print("FINAL SELECTED VARIABLES")
print("="*80)
print(final_selected_vars)
print(f"\nTotal selected variables: {len(final_selected_vars)}")

# =========================================================
# 8. 生成最终建模数据
# =========================================================

model_df = analysis_df[final_selected_vars + [target_col]].copy()

print("\n=== Final model_df shape ===")
print(model_df.shape)
print(model_df.head())

# =========================================================
# 9. 输出结果表到 CSV
# =========================================================

import os

output_dir = os.getcwd()   # 也可改成你自己的路径
print("\nCSV output directory:", output_dir)

# 9.1 转换日志
transformation_log_df.to_csv(os.path.join(output_dir, "transformation_log.csv"), index=False)

# 9.2 每个区块的 MI 结果
block_mi_output = []
for block_name, res in selection_results.items():
    if not res["mi_table"].empty:
        tmp = res["mi_table"].copy()
        tmp["block"] = block_name
        tmp["selected_in_block"] = tmp["variable"].isin(res["selected"])
        tmp["dropped_high_missing"] = tmp["variable"].isin(res["dropped_high_missing"])
        tmp["dropped_low_mi"] = tmp["variable"].isin(res["dropped_low_mi"])
        tmp["dropped_high_corr"] = tmp["variable"].isin(res["dropped_high_corr"])
        block_mi_output.append(tmp)

if len(block_mi_output) > 0:
    block_mi_output_df = pd.concat(block_mi_output, axis=0, ignore_index=True)
else:
    block_mi_output_df = pd.DataFrame()

block_mi_output_df.to_csv(
    os.path.join(output_dir, "block_screening_results_xy_withinblockcorr.csv"),
    index=False
)

# 9.3 最终保留变量表
final_selected_df = pd.DataFrame({
    "variable": final_selected_vars
})
final_selected_df.to_csv(
    os.path.join(output_dir, "final_selected_variables.csv"),
    index=False
)

# 9.4 区块汇总表
selected_summary = pd.DataFrame({
    "block": list(selection_results.keys()),
    "selected_variables": [selection_results[b]["selected"] for b in selection_results.keys()],
    "n_selected": [len(selection_results[b]["selected"]) for b in selection_results.keys()]
})
selected_summary.to_csv(
    os.path.join(output_dir, "selected_summary_by_block.csv"),
    index=False
)

print("\nSaved files:")
print("- transformation_log.csv")
print("- block_screening_results_xy_withinblockcorr.csv")
print("- final_selected_variables.csv")
print("- selected_summary_by_block.csv")

=== Shape after dropping missing target ===
(13036, 93)

=== Transformation log ===
   original_variable used_variable            transformation
0                CB3           CB3                       raw
1                CB4           CB4  raw_non_numeric_or_empty
2               CB5A          CB5A  raw_non_numeric_or_empty
3               CB5B          CB5B                       raw
4                CB7           CB7  raw_non_numeric_or_empty
..               ...           ...                       ...
77               WS7           WS7  raw_non_numeric_or_empty
78              WS11          WS11  raw_non_numeric_or_empty
79              WS14          WS14  raw_non_numeric_or_empty
80              WS15          WS15  raw_non_numeric_or_empty
81               HW5           HW5  raw_non_numeric_or_empty

[82 rows x 3 columns]

=== Transformed blocks ===
child_background : ['CB3', 'CB4', 'CB5A', 'CB5B', 'CB7', 'CB11', 'HH6', 'HH7', 'HL4', 'ethnicity', 'wscore_log1p']
child_labour : ['C

# Model Fitting and Tuning

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. As with figures, any included code must also be addressed in the text of the document.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

# Interpretation, Discussion & Conclusions

*In this section you should provide a general overview of your final model, its performance, and reliability. You should discuss what the implications of your model are in terms of the included features, estimated parameters and relationships, predictive performance, and anything else you think is relevant.*

*This should be written with a target audience of a government official, who understands the issues associated with mental health but may only have university level mathematics (not necessarily postgraduate statistics or machine learning). Your goal should be to highlight to this audience how your model can useful. You should also discuss potential limitations or directions of future improvement of your model.*

*Finally, you should include recommendations on factors that may increase the risk of depression, which may be useful for the government officials and health care workers to improve their understanding of the condition, and potentially assit in the development of effective social and health policies and interventions.*

*Keep in mind that a negative result, i.e. a model that does not work well predictively, that is well explained and justified in terms of why it failed will likely receive higher marks than a model with strong predictive performance but with poor or incorrect explanations / justifications.*

# Generative AI statement

*Include a statement on how generative AI was used in the project and report.*

# References

*Include references if any*

In [ ]:
# Run the following to render to PDF
!jupyter nbconvert --to pdf project.ipynb